# 📈 Notebook 2: The Phi (φ) Accrual Failure Detector

Notebook 1 left us with a dilemma: a single timeout is either too jumpy or too slow, and the "right" value depends on the network, the workload, even the time of day.

**Hayashibara et al. (2004)** proposed a radically different interface:

> Instead of returning *alive / dead*, output a continuous **suspicion value** `φ` that grows the longer we go without a heartbeat.

Each caller picks its own threshold: cache routers may act at `φ ≈ 3`; leader fencing may wait for `φ ≈ 12`. The detector itself makes **no decision** — it just gives you a number that means *"how surprised am I that I haven't heard from this node yet?"*

This is the algorithm used by **Apache Cassandra**, **Akka Cluster**, and **ScyllaDB**.


## 1. The math, one line at a time

Keep a sliding window of the last `N` inter-arrival times (e.g. the last 200 heartbeat gaps). Treat them as samples from a Normal distribution with mean `μ` and std-dev `σ` (Cassandra/Akka approximation — the original paper uses an exponential).

Let `Δ = t_now − t_last_heartbeat` be the current silence.

1. Compute the tail probability `P(interval ≥ Δ)` — *"how likely is it that a real interval would be at least this long?"*
2. Squeeze it to a more human scale: `φ = -log₁₀(P)`.

Quick feel for the numbers:

| `φ` | `P(interval ≥ Δ)` ≈ | Meaning |
|---:|---:|---|
| 1  | 10⁻¹ (10%)   | Slightly late — nothing to worry about |
| 3  | 10⁻³ (0.1%)  | Getting suspicious |
| 8  | 10⁻⁸         | **Cassandra's default — act now** |
| 12 | 10⁻¹²        | **Akka's default — absolute certainty** |

The survival function of a Normal distribution is `P(X ≥ Δ) = ½ · erfc((Δ−μ) / (σ·√2))`. Python ships `math.erfc` so we can compute it in one line.


## 2. Implementation

Tiny, dependency-free, commented so you can port it anywhere.


In [ ]:
import math
from collections import deque

class PhiAccrual:
    """Phi accrual failure detector (Normal approximation, the Cassandra flavor).

    Usage:
        d = PhiAccrual()
        d.heartbeat(t)    # whenever a heartbeat arrives
        d.phi(now)        # any time you want the current suspicion
    """

    def __init__(self, window=200, min_std=0.1, warmup=10):
        # sliding window of recent inter-arrival gaps
        self.intervals = deque(maxlen=window)
        self.last = None
        # floor on std-dev: on super-stable links the measured std can be tiny,
        # which makes phi explode on even a sliver of lateness. Cassandra uses 0.1s.
        self.min_std = min_std
        # need at least this many samples before we trust our statistics
        self.warmup = warmup

    def heartbeat(self, t: float) -> None:
        if self.last is not None:
            self.intervals.append(t - self.last)
        self.last = t

    def phi(self, now: float) -> float:
        # Not enough history yet -> no opinion (avoids cold-start false alarms)
        if self.last is None or len(self.intervals) < self.warmup:
            return 0.0
        mean = sum(self.intervals) / len(self.intervals)
        var  = sum((x - mean) ** 2 for x in self.intervals) / len(self.intervals)
        std  = max(math.sqrt(var), self.min_std)

        delta = now - self.last
        z     = (delta - mean) / std
        # P(interval >= delta) under Normal(mean, std)
        p     = 0.5 * math.erfc(z / math.sqrt(2))
        # clamp to avoid log10(0) when delta is enormous (keeps phi finite)
        p     = max(p, 1e-20)
        return -math.log10(p)

## 3. Replay the same scenario from notebook 1

Same jittery heartbeat stream, same crash at `t = 20s`. This time we watch `φ` over time.

In [ ]:
import random, matplotlib.pyplot as plt

random.seed(7)
INTERVAL, JITTER, TOTAL, DEAD_AT = 1.0, 0.4, 30.0, 20.0

def make_trace(dead_at=DEAD_AT, total=TOTAL, blip=None):
    beats, now = [], 0.0
    while now < total:
        now += INTERVAL + random.uniform(-JITTER, JITTER)
        if now >= dead_at:
            break
        if blip is not None and blip[0] <= now < blip[0] + blip[1]:
            continue  # pretend the heartbeat was dropped by the network
        beats.append(now)
    return beats

heartbeats = make_trace()

def run_phi(beats, total=TOTAL, step=0.05):
    det = PhiAccrual()
    ts, phis = [], []
    i, t = 0, 0.0
    while t < total:
        while i < len(beats) and beats[i] <= t:
            det.heartbeat(beats[i]); i += 1
        ts.append(t); phis.append(det.phi(t))
        t += step
    return ts, phis

ts, phis = run_phi(heartbeats)

plt.figure(figsize=(10, 3.5))
plt.plot(ts, phis, label='φ')
plt.axhline(1,  color='gold',    linestyle=':', label='φ=1  (mild)')
plt.axhline(8,  color='tab:red', linestyle='--', label='φ=8  (Cassandra)')
plt.axhline(12, color='purple',  linestyle='-.', label='φ=12 (Akka)')
plt.axvline(DEAD_AT, color='black', linestyle=':', label='real crash')
plt.plot(heartbeats, [0.2]*len(heartbeats), '|', color='tab:green', label='heartbeat')
plt.xlabel('time (s)'); plt.ylabel('φ'); plt.legend(loc='upper left', fontsize=8)
plt.title('Phi stays near zero during healthy jitter and rockets up after the real crash')
plt.show()

for th in (1, 8, 12):
    cross = next((t for t, p in zip(ts, phis) if p > th), None)
    print(f'threshold φ={th:>2}: crossed at t={cross:.2f}s  ({cross - DEAD_AT:+.2f}s vs real crash)' if cross else f'threshold φ={th}: never crossed')

## 4. The killer feature: surviving a network blip

Real networks lose bursts of packets. Imagine we drop **every** heartbeat for 1.5 seconds in the middle of the node's life, but the node is perfectly alive.

A fixed 1.0-second timeout would scream FAILURE (a classic split-brain trigger). Phi will get **suspicious** — it *should* — but if the blip ends before we cross the threshold, no action is taken and `φ` collapses back to ~0.

In [ ]:
random.seed(7)
blip_trace = make_trace(dead_at=TOTAL, blip=(15.0, 1.5))  # node never actually dies

ts_b, phis_b = run_phi(blip_trace)

peak = max(p for t, p in zip(ts_b, phis_b) if 15.0 <= t <= 17.0)
print(f'peak φ during 1.5s blip: {peak:.2f}')
print(f'would a Cassandra detector (φ>8) false-alarm? {"YES" if peak > 8 else "NO"}')
print(f'would a fixed 1.0s timeout false-alarm?      YES (blip is 1.5s long)')

plt.figure(figsize=(10, 3.5))
plt.plot(ts_b, phis_b, label='φ')
plt.axvspan(15.0, 16.5, color='grey', alpha=0.25, label='network blip')
plt.axhline(8, color='tab:red', linestyle='--', label='φ=8')
plt.xlabel('time (s)'); plt.ylabel('φ'); plt.legend()
plt.title('Phi rises during the blip, then drops back — no false alarm')
plt.show()

## 5. Side-by-side vs fixed timeout

One picture, same trace with the blip:

In [ ]:
def fixed_timeout_flags(beats, timeout, total=TOTAL, step=0.05):
    times, flags = [], []
    last_seen = 0.0; i = 0; t = 0.0
    while t < total:
        while i < len(beats) and beats[i] <= t:
            last_seen = beats[i]; i += 1
        times.append(t)
        flags.append(1 if (t - last_seen) > timeout else 0)
        t += step
    return times, flags

t_ft, f_ft = fixed_timeout_flags(blip_trace, timeout=1.0)

fig, (a, b) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
a.fill_between(t_ft, 0, f_ft, step='pre', color='tab:red', alpha=0.4)
a.axvspan(15.0, 16.5, color='grey', alpha=0.25)
a.set_title('Fixed 1.0s timeout: false DOWN for the entire blip 🚨')
a.set_yticks([0,1]); a.set_ylabel('DOWN?')

b.plot(ts_b, phis_b)
b.axhline(8, color='tab:red', linestyle='--')
b.axvspan(15.0, 16.5, color='grey', alpha=0.25)
b.set_title('Phi accrual on the same trace: suspicious, but never crosses 8 ✅')
b.set_xlabel('time (s)'); b.set_ylabel('φ')
plt.tight_layout(); plt.show()

## 6. Why phi wins — summary

|                       | Fixed timeout             | Phi accrual                                   |
|-----------------------|---------------------------|-----------------------------------------------|
| Adapts to network     | ❌ you hand-tune           | ✅ learns μ and σ from the window             |
| One value fits all    | ❌ every app shares it     | ✅ each caller picks its threshold            |
| False positives       | many under jitter         | rare; suspicion grows smoothly                |
| Crash detection speed | fixed by the timeout      | fast when network is calm, patient when noisy |

**Practical tips**

- `window ≈ 100–1000` recent intervals is plenty.
- Floor the std-dev (`min_std ≈ 0.1s`). Without it, an unnaturally stable link makes `φ` explode on the tiniest lateness.
- Don't mix short-lived bursts (like gossip pings) with the steady-state window. Reset the detector on reconnect.
- Phi itself is **passive**. What to *do* when it crosses your threshold (back off traffic, start a replacement, trigger leader election) is the topic of notebook 3.

👉 Continue with `03_real_world_cluster.ipynb`.